# Reasoning Techniques (Deep Research)

The Deep Research pattern implements the **generate → research → reflect → finalize** loop: an agent generates search queries, gathers information, reflects on knowledge gaps, and iteratively refines until it can synthesize a comprehensive answer.

This is the architecture behind tools like Perplexity, Google Deep Research, and the open-source `gemini-fullstack-langgraph-quickstart`.

## Implementation with Flyte v2 + the Agent harness

This refactor subclasses `Agent` as a `DeepResearchAgent` whose `run` walks the four phases explicitly, calling `super().run.aio(...)` for each. The *research* phase hands the model a `web_search` tool and lets the inherited loop drive the tool calls; the other phases are tool-free reasoning steps. The LangGraph `StateGraph` and its conditional edges become a plain Python `for` loop inside the subclass.

#### LangGraph vs Flyte v2 + Agent harness

| Aspect | LangGraph | Flyte v2 + `Agent` harness |
|--------|-----------|----------------------------|
| **Graph definition** | `StateGraph` + `add_conditional_edges` | `DeepResearchAgent.run` — a Python loop |
| **Research step** | `web_research` node | `web_search` `@env.task` tool, driven by `super().run.aio` |
| **Conditional routing** | `evaluate_research` edge | `if sufficient: break` |
| **Checkpointing** | LangGraph checkpointer | Tools traced as nested actions in the UI |
| **Live progress** | LangSmith traces | `flyte.report` HTML tab, updated each iteration |
| **Secrets** | `.env` / `os.environ` | `flyte.Secret` injected by cluster |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' litellm duckduckgo-search

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

### 3. Import dependencies and configure the TaskEnvironment

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from datetime import timedelta

import flyte
import flyte.report
from flyte.ai.agents import Agent, AgentResult
from flyte.syncify import syncify

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="reasoning-agent", python_version=(3, 12))
    .with_pip_packages("litellm", "duckduckgo-search>=6.0.0")
)

reasoning_env = flyte.TaskEnvironment(
    name="reasoning_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define the result model

`ResearchResult` is the typed output surfaced in the Flyte UI: the question, the synthesized answer, and how many research iterations it took. The intermediate state (queries, findings, knowledge gaps) is managed inside the agent's `run` loop rather than threaded through a `StateGraph`.

In [ ]:
@dataclass
class ResearchResult:
    """Final output of the Deep Research agent."""
    question: str
    answer: str
    iterations: int

### 5. Define the `web_search` tool and `DeepResearchAgent`

`web_search` is an `@env.task` tool, so each search appears as a nested, traced action in the UI. `DeepResearchAgent` overrides `run` to walk the four phases — generate queries, research (via the tool), reflect, finalize — delegating each to `super().run.aio(...)`. Only the research phase is told to use the tool; the others are tool-free reasoning prompts.

In [ ]:
@reasoning_env.task(cache="auto")
async def web_search(query: str) -> str:
    """Search the web and return the top results as text.

    Args:
        query: The search query.
    """
    from duckduckgo_search import DDGS

    results: list[str] = []
    try:
        for hit in (DDGS().text(query, max_results=3) or []):
            results.append(
                f"[{hit.get('title', '')}] {hit.get('body', '')} ({hit.get('href', '')})"
            )
    except Exception as exc:  # network/ratelimit — surface to the model, don't crash
        return f"Search failed: {exc}"
    return "\n".join(results) if results else "No results found."


def _esc(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


@dataclass
class DeepResearchAgent(Agent):
    """generate -> research -> reflect -> finalize, each phase via super().run.aio."""
    max_iterations: int = 3

    @syncify
    async def run(self, message: str, history: list | None = None) -> AgentResult:
        question = message
        gaps: list[str] = []
        findings: list[str] = []
        used = 0

        for i in range(self.max_iterations):
            used = i + 1

            # Phase 1 — generate queries (no searching yet).
            gap_text = ("\nKnowledge gaps to address:\n" + "\n".join(gaps)) if gaps else ""
            q = await super(DeepResearchAgent, self).run.aio(
                f"Research question: {question}{gap_text}\n"
                "List 2-3 focused web search queries, one per line. Do NOT search yet."
            )
            queries = [ln.strip() for ln in (q.summary or "").splitlines() if ln.strip()][:3]

            # Phase 2 — research: let the inherited loop drive the web_search tool.
            r = await super(DeepResearchAgent, self).run.aio(
                "Use the web_search tool to gather facts for each of these queries, then "
                "summarize what you found:\n" + "\n".join(queries)
            )
            findings.append(r.summary or "")

            # Phase 3 — reflect: sufficient? remaining gaps?
            refl = await super(DeepResearchAgent, self).run.aio(
                f"Question: {question}\n\nFindings so far:\n" + "\n\n".join(findings)
                + "\n\nReply line 1: SUFFICIENT or INSUFFICIENT. Lines 2+: remaining "
                "knowledge gaps, one per line. Do NOT search."
            )
            lines = (refl.summary or "").strip().splitlines()
            sufficient = bool(lines) and lines[0].strip().upper().startswith("SUFF")
            gaps = [ln.lstrip("- ").strip() for ln in lines[1:] if ln.strip()]

            await flyte.report.log.aio(
                f"<h3>Iteration {used} — {'SUFFICIENT' if sufficient else 'needs more'}</h3>"
                f"<p><strong>Queries:</strong> {_esc(', '.join(queries))}</p>"
                f"<pre>{_esc(findings[-1])}</pre>"
            )
            await flyte.report.flush.aio()

            if sufficient:
                break

        # Phase 4 — finalize.
        final = await super(DeepResearchAgent, self).run.aio(
            f"Question: {question}\n\nAll findings:\n" + "\n\n".join(findings)
            + "\n\nWrite a comprehensive, well-structured final answer. Do NOT search."
        )
        return AgentResult(summary=final.summary or "", attempts=used)

### 6. Define the Deep Research orchestrator task

#### From `StateGraph` to a subclassed Agent

The earlier version compiled the loop by hand: `@flyte.trace`-d `_generate_queries` / `_web_research` / `_reflection` helpers, a `ResearchState` dataclass threaded across nodes, and an `if state.sufficient: break` written into the task. All of that now lives in `DeepResearchAgent.run`; the task just constructs the agent and maps the `AgentResult`.

In [ ]:
researcher = DeepResearchAgent(
    name="deep-researcher",
    model="claude-sonnet-4-6",
    instructions=(
        "You are a meticulous research assistant. When asked to research, use the "
        "web_search tool to gather facts before answering. Cite what you found."
    ),
    tools=[web_search],
    max_iterations=3,
)


@reasoning_env.task(
    retries=2,
    timeout=timedelta(minutes=20),
    cache=flyte.Cache(behavior="disable"),
    report=True,
)
async def deep_research(question: str, max_iterations: int = 3) -> ResearchResult:
    """Deep Research: generate queries -> search -> reflect -> finalize.

    The four-phase loop is owned by DeepResearchAgent.run; each web_search call shows
    up as a nested action, and each iteration is logged live to the report tab.
    """
    researcher.max_iterations = max_iterations
    result: AgentResult = await researcher.run.aio(question)
    return ResearchResult(
        question=question,
        answer=result.summary,
        iterations=result.attempts,
    )

### 7. Run locally

In [ ]:
run = flyte.run(
    deep_research,
    question="What are the main differences between Flyte and Apache Airflow for ML pipelines?",
    max_iterations=3,
)
run.wait()
result = run.outputs()[0]
print(f"Iterations: {result.iterations}")
print(result.answer)

### Running remotely

Remote execution adds the live `report` tab — watch each iteration's queries, results, and knowledge gaps update in real time as the agent researches.

In [ ]:
run = flyte.run(
    deep_research,
    question="What are the main differences between Flyte and Apache Airflow for ML pipelines?",
    max_iterations=3,
)
run.wait()
result = run.outputs()[0]
print(f"Iterations: {result.iterations}")
print(result.answer)

## Scaling the pattern

The devbox runs each task in a fresh container. For production workloads with many short LLM calls, `ReusePolicy` eliminates cold-start overhead by keeping a pool of warm containers ready.

> **Note:** `ReusePolicy` is a Union-specific feature that requires a [Union deployment](https://www.union.ai/docs/v2/union/). It is not supported on the local devbox.

In [ ]:
# Requires a Union deployment — not supported on the local devbox
from datetime import timedelta

production_reasoning_agent = flyte.TaskEnvironment(
    name="reasoning_agent_prod",
    image=_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    secrets=[flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")],
    reusable=flyte.ReusePolicy(
        replicas=(2, 8),
        concurrency=4,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=15),
    ),
)